In [2]:
import polars as pl

In [3]:
#TODO: decide if this check is within the function or not.(Function only has to be called once).

df = pl.DataFrame()
df = pl.read_csv('plz_geocoord.csv', dtypes={"plz": pl.Utf8}) # Ensures that the plz are strings.
print(df)

shape: (8_298, 3)
┌───────┬───────────┬───────────┐
│ plz   ┆ lat       ┆ lng       │
│ ---   ┆ ---       ┆ ---       │
│ str   ┆ f64       ┆ f64       │
╞═══════╪═══════════╪═══════════╡
│ 01067 ┆ 51.05755  ┆ 13.717065 │
│ 01069 ┆ 51.039135 ┆ 13.737675 │
│ 01097 ┆ 51.065908 ┆ 13.736152 │
│ 01099 ┆ 51.087188 ┆ 13.802804 │
│ 01108 ┆ 51.144324 ┆ 13.799706 │
│ …     ┆ …         ┆ …         │
│ 99988 ┆ 51.161168 ┆ 10.239124 │
│ 99991 ┆ 51.152222 ┆ 10.574434 │
│ 99994 ┆ 51.244929 ┆ 10.659656 │
│ 99996 ┆ 51.300396 ┆ 10.574358 │
│ 99998 ┆ 51.231192 ┆ 10.587336 │
└───────┴───────────┴───────────┘


/var/folders/y2/4y36_h4x2717m_5x0dk9_7yw0000gn/T/ipykernel_15792/830335312.py:4: DeprecationWarning: the argument `dtypes` for `read_csv` is deprecated. It was renamed to `schema_overrides` in version 0.20.31.
  df = pl.read_csv('plz_geocoord.csv', dtypes={"plz": pl.Utf8}) # Ensures that the plz are strings.


In [4]:
#NOTE: function assumes that df has a column call "plz"
def get_leitregionen_coords_df(plz_df:pl.DataFrame):
    plz_df = plz_df.with_columns(
    pl.col("plz").str.slice(0, 2).alias("plz")
    ) #->with help of chatgpt.

    plz_df = plz_df.group_by("plz").agg(pl.col("lat").mean(), # with help of chatgpt
                                        pl.col("lng").mean()
                                        ).sort(pl.col("plz"), descending=False)
    
    plz_df = plz_df.rename({"lat": "avg_lat"}
                           ).rename({"lng": "avg_lng"}
                                    ).rename({"plz": "leit_plz"})    

    return plz_df




In [5]:
# Im Prinzip kann man überlegen, ob dieser df nicht als Konstante gespeichert werden könnte. Da ändert sich ja sowieso nichts.
plz_df = get_leitregionen_coords_df(df)

plz_df.write_csv('plz_leitregionen.csv')

print(plz_df)

shape: (96, 3)
┌──────────┬───────────┬───────────┐
│ leit_plz ┆ avg_lat   ┆ avg_lng   │
│ ---      ┆ ---       ┆ ---       │
│ str      ┆ f64       ┆ f64       │
╞══════════╪═══════════╪═══════════╡
│ 01       ┆ 51.111769 ┆ 13.768566 │
│ 02       ┆ 51.154014 ┆ 14.61334  │
│ 03       ┆ 51.701574 ┆ 13.980449 │
│ 04       ┆ 51.297898 ┆ 12.593077 │
│ 06       ┆ 51.513763 ┆ 11.799856 │
│ …        ┆ …         ┆ …         │
│ 95       ┆ 50.08067  ┆ 11.803764 │
│ 96       ┆ 50.085047 ┆ 10.995421 │
│ 97       ┆ 49.936608 ┆ 9.983503  │
│ 98       ┆ 50.58834  ┆ 10.731629 │
│ 99       ┆ 51.071354 ┆ 10.843627 │
└──────────┴───────────┴───────────┘


In [6]:
other_path = '/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/stations.csv'

df2 = pl.DataFrame()
df2 = pl.read_csv(source=other_path, dtypes={"post_code": pl.Utf8})
df2 = df2.rename({"post_code": "plz"}).rename({"latitude": "lat"}).rename({"longitude": "lng"})
print(df2)

shape: (15_442, 9)
┌────────────┬────────────┬────────────┬───────────┬───┬───────┬───────────┬───────────┬───────────┐
│ uuid       ┆ name       ┆ brand      ┆ street    ┆ … ┆ plz   ┆ city      ┆ lat       ┆ lng       │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---   ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ str        ┆ str        ┆ str       ┆   ┆ str   ┆ str       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════╪═══════════╪═══════════╪═══════════╡
│ 00060723-0 ┆ BAGeno     ┆            ┆ Künzelsau ┆ … ┆ 74653 ┆ Ingelfing ┆ 49.296822 ┆ 9.661385  │
│ 001-4444-8 ┆ Raiffeisen ┆            ┆ er        ┆   ┆       ┆ en        ┆           ┆           │
│ 888-acdc00 ┆ eG         ┆            ┆ Strasse   ┆   ┆       ┆           ┆           ┆           │
│ …          ┆            ┆            ┆           ┆   ┆       ┆           ┆           ┆           │
│ 005056ba-7 ┆ famila     ┆ FAMILA     ┆ Pascalstr ┆ … ┆ 25442 ┆ Quickbo

/var/folders/y2/4y36_h4x2717m_5x0dk9_7yw0000gn/T/ipykernel_15792/1521777193.py:4: DeprecationWarning: the argument `dtypes` for `read_csv` is deprecated. It was renamed to `schema_overrides` in version 0.20.31.
  df2 = pl.read_csv(source=other_path, dtypes={"post_code": pl.Utf8})


In [7]:
df2_coordinate_avg = get_leitregionen_coords_df(df2)

print(df2_coordinate_avg)
print(plz_df)

shape: (99, 3)
┌──────────┬───────────┬───────────┐
│ leit_plz ┆ avg_lat   ┆ avg_lng   │
│ ---      ┆ ---       ┆ ---       │
│ str      ┆ f64       ┆ f64       │
╞══════════╪═══════════╪═══════════╡
│          ┆ 25.340651 ┆ 4.074061  │
│ 00       ┆ 50.722975 ┆ 7.1146229 │
│ 01       ┆ 51.154641 ┆ 13.758695 │
│ 02       ┆ 51.219612 ┆ 14.54748  │
│ 03       ┆ 51.730483 ┆ 14.152932 │
│ …        ┆ …         ┆ …         │
│ 97       ┆ 49.902361 ┆ 10.014935 │
│ 98       ┆ 50.597056 ┆ 10.690124 │
│ 99       ┆ 51.055146 ┆ 10.840169 │
│ Ni       ┆ 49.781785 ┆ 11.189532 │
│ ni       ┆ 53.306463 ┆ 7.524143  │
└──────────┴───────────┴───────────┘
shape: (96, 3)
┌──────────┬───────────┬───────────┐
│ leit_plz ┆ avg_lat   ┆ avg_lng   │
│ ---      ┆ ---       ┆ ---       │
│ str      ┆ f64       ┆ f64       │
╞══════════╪═══════════╪═══════════╡
│ 01       ┆ 51.111769 ┆ 13.768566 │
│ 02       ┆ 51.154014 ┆ 14.61334  │
│ 03       ┆ 51.701574 ┆ 13.980449 │
│ 04       ┆ 51.297898 ┆ 12.593077 │
│ 06    